# 01: Data Preparation

This notebook prepares data from the canonical **data blocks pool** for upcoming training / evaluation sessions:
- feature channel selection and target (training head) schemes
- train / validation / test partition
- per-band statistics and normalization into prepared `.npz` blocks

> **Prerequisite**: an existing ingestion output (data blocks pool), e.g. from `01_data_ingestion`. Point widget `01` at it; the feature and target dropdowns are built from that pool.

## Environment Setup & Imports

> **Prerequisite**: Ensure this notebook is attached to a cluster with the `landseg` wheel pre-installed (e.g. Databricks compute via cluster init script or cluster libraries) or an active local virtual environment. No in-notebook pip installation is required.

In [ ]:
# standard imports
import json
import pathlib
import sys
# third-party imports
import numpy
# local imports
# Attempt importing landseg directly from site-packages (wheel install).
# Fall back to local '../src' only when running inside the repository clone.
try:
    import landseg.adapters.api as api
    import landseg.runtime as runtime
    import landseg.utils.validator as validator
except ImportError:
    repo_src = pathlib.Path.cwd().parent / 'src'
    if repo_src.exists():
        sys.path.insert(0, str(repo_src))
    import landseg.adapters.api as api
    import landseg.runtime as runtime
    import landseg.utils.validator as validator

## Configuration

This notebook is independent of where the raw data came from: it only needs an **ingestion root** (input) and a **prepared output dir**. Both default to the `01_data_ingestion` default locations; the notebook stops with a clear message if either is not set or the ingestion root has no data blocks pool. Settings are widgets, grouped by number:

| Widgets | Group | Notes |
| --- | --- | --- |
| `01` | Input | Ingestion root (required); must contain `data_blocks/schema.json`. `01_data_ingestion` prints this path in its pool report. |
| `02` | Output | Prepared output dir (required); must differ from the ingestion root. |
| `11-13` | Preparation settings | Rebuild flag, validation / test split ratios. |
| `21+` | Feature schemes | One dropdown per feature dataset with schemes in the pool (`all` keeps every band). |
| `31+` | Training heads | One dropdown per training head in the pool (`raw` keeps canonical classes). |

In [ ]:
# set up widgets (zero-padded numbered names keep them grouped and ordered in the widget bar)
widgets = runtime.NotebookWidgets()

# 01-02 I/O: ingestion root (data blocks pool) in, prepared data out; defaults match 01_data_ingestion defaults
ingest_root = widgets.text('01_ingest_root', default='../experiment/artifacts/ingested_data', label='01. Ingestion Root (input)')
prep_output = widgets.text('02_prep_output', default='../experiment/artifacts/prepared_data', label='02. Prepared Output Dir')

# 11-13 preparation settings
prep_rebuild = widgets.boolean('11_prep_rebuild', default=True, label='11. Force Rebuild')
val_ratio = widgets.float('12_val_ratio', default=0.20, label='12. Validation Ratio')
test_ratio = widgets.float('13_test_ratio', default=0.00, label='13. Test Holdout Ratio')

In [ ]:
# 21+ / 31+ feature scheme and training head dropdowns are built from the pool schema
# 01-02. I/O must be set explicitly (defaults point at 01_data_ingestion outputs)
missing = [
    label for label, val in [
        ('01. Ingestion Root (input)', ingest_root),
        ('02. Prepared Output Dir', prep_output),
    ] if not val.strip()
]
if missing:
    raise ValueError(f'Required widget(s) not set: {", ".join(missing)}')
resolved_ingest_dpath = pathlib.Path(ingest_root.strip()).expanduser()
resolved_prep_dpath = pathlib.Path(prep_output.strip()).expanduser()

# 01. the ingestion root must hold a data blocks pool
try:
    pool_schema_fpath = validator.require_file(resolved_ingest_dpath / 'data_blocks' / 'schema.json')
except validator.ValidationError as err:
    raise ValueError(
        f'No data blocks pool found at {resolved_ingest_dpath / "data_blocks"}. '
        'Run 01_data_ingestion first, or set widget "01. Ingestion Root" to an existing ingestion output.'
    ) from err

# 02. prepared output must not overwrite the ingestion root
if resolved_prep_dpath.resolve() == resolved_ingest_dpath.resolve():
    raise ValueError('Widget "02. Prepared Output Dir" must differ from "01. Ingestion Root".')

# 12-13. split ratios must leave blocks for training
if not (0 <= val_ratio < 1 and 0 <= test_ratio < 1 and val_ratio + test_ratio < 1):
    raise ValueError(f'Invalid split ratios: validation={val_ratio}, test={test_ratio} (each in [0, 1), sum < 1).')

# log resolved configuration
print(f'Data blocks pool:   {pool_schema_fpath.parent}')
print(f'Prepared output:    {resolved_prep_dpath}')
print(f'Rebuild:            {prep_rebuild}')
print(f'Split val / test:   {val_ratio} / {test_ratio}')

In [ ]:
# dropdown choices come from the data blocks pool schema (see the report in 01_data_ingestion)
pool_schema = json.loads(pool_schema_fpath.read_text(encoding='utf-8'))
image_schemes = pool_schema['dataset'].get('image_schemes', {})
label_schemes = pool_schema['dataset'].get('label_schemes', {})
training_heads = list(pool_schema['io_conventions']['label_band_map'])

# preferred defaults for the shipped dummy datasets (ignored if not in a dataset's choices)
DEMO_DEFAULTS = {'sentinel2': 'rgb_nir', 'landcover': 'binary'}

def _default(name: str, choices: list[str]) -> str:
    return DEMO_DEFAULTS[name] if DEMO_DEFAULTS.get(name) in choices else choices[0]

# 21+ feature schemes: one dropdown per feature dataset with schemes ('all' keeps every band)
feature_schemes = {}
for n, (ds_name, schemes) in enumerate(image_schemes.items(), start=21):
    choices = ['all', *schemes]
    feature_schemes[ds_name] = widgets.dropdown(
        f'{n}_features_{ds_name}', _default(ds_name, choices), choices, label=f'{n}. Features | {ds_name}'
    )

# 31+ training heads: one dropdown per head ('raw' keeps canonical classes)
target_schemes = {}
for n, head in enumerate(training_heads, start=31):
    choices = ['raw', *label_schemes.get(head, {})]
    target_schemes[head] = widgets.dropdown(
        f'{n}_targets_{head}', _default(head, choices), choices, label=f'{n}. Targets | {head}'
    )

print(f'Feature schemes: {feature_schemes}')
print(f'Target schemes:  {target_schemes}')

## Run Data Preparation

Selects feature channels, resolves target schemes into multi-head targets, partitions blocks into train / val / test, calculates per-band statistics, and normalizes them into prepared `.npz` blocks. Re-run after changing the widgets above to prepare data for another session.


In [ ]:
# data preparation
prep_configurator = api.DataPreparationConfigurator(
    upstream_dpath=resolved_ingest_dpath,
    output_dpath=resolved_prep_dpath,
)

# feature channel selection and target reclassification schemes (widgets 21+ / 31+)
prep_configurator.set_features(feature_schemes)
prep_configurator.set_targets(target_schemes)

# partition ratios (widgets 12-13); remaining blocks are used for training
prep_configurator.set_partition(
    validation_blocks_ratio=val_ratio,
    test_holdout_blocks_ratio=test_ratio,
)

# force rebuild preparation artifacts (widget 11)
prep_configurator.set_rebuild(prep_rebuild)

# run data preparation pipeline
api.run(prep_configurator.running_root_config)


## Inspect Prepared Artifacts & Dataset Schema

Let's inspect the compiled `schema.json` and a sample prepared `.npz` data block:

In [ ]:
# 1. Read Dataset Transform Schema
schema_path = resolved_prep_dpath / 'schema.json'
with open(schema_path, 'r', encoding='UTF-8') as f:
    schema = json.load(f)

print("=== Dataset Transform Schema ===")
print(f"Train Blocks: {len(schema['train_blocks'])}")
print(f"Val Blocks:   {len(schema['val_blocks'])}")
print(f"Test Blocks:  {len(schema['test_blocks'])}")
print("\nImage Channels Normalization Stats (Mean / Std):")
for band, stats in schema['image_stats'].items():
    print(f"  {band}: mean={stats['current_mean']:.4f}, std={stats['std']:.4f}")

# 2. Inspect a sample prepared block
_, sample_block_path = next(iter(schema['train_blocks'].items()))
sample_block_path = pathlib.Path(sample_block_path)
block_data = numpy.load(sample_block_path)

print(f"\n=== Sample Block ({sample_block_path.name}) ===")
print(f"Keys: {list(block_data.keys())}")
print(f"Image Array Shape (C, H, W): {block_data['image'].shape}")
print(f"Label Array Shape (L, H, W): {block_data['label'].shape}")
print(f"Unique Label Values in Base Head: {numpy.unique(block_data['label'][0])}")